# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahmanmohamed05/FlyRank-AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**This is a scoring task.**

My lane is Refresh / Content Opportunity Scoring: for every page, I want a number that says how much "opportunity" there is in refreshing it, so an editor can sort the whole site by that number and work down the list.

- **Not classification** — refresh-worthiness isn't a clean yes/no. A page that's mildly overdue and a page that's badly overdue and losing traffic are both "worth refreshing," but not equally. Forcing that into two buckets throws away the information an editor actually needs (which page goes first).
- **Not clustering** — I already know what I'm grouping by (opportunity), I'm not trying to discover unknown groupings in the pages.
- **Not ranking (as the primary framing)** — a score naturally produces a ranking once you sort by it, so ranking comes along for free. But the score itself is the deliverable: it needs to mean something on its own ("this page scores 82/100"), not just say "this page is above that one." That's the difference between scoring and ranking, and scoring is the better fit here.

In [1]:
TASK_TYPE = "scoring"
print(f"Lane: Refresh / Content Opportunity Scoring")
print(f"ML task type: {TASK_TYPE}")


Lane: Refresh / Content Opportunity Scoring
ML task type: scoring


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**My proxy: staleness combined with traffic potential.**

There's no column in the data literally called "opportunity" — nobody labeled these pages "refresh me" or "leave me alone." So this has to be a **proxy target**, built from signals that are actually in the data, not an observed outcome.

Staleness alone isn't enough: an old page that gets no traffic and ranks nowhere isn't an "opportunity," it's just old. And traffic/ranking potential alone isn't enough either: a page that's already fresh and performing well doesn't need a refresh no matter how much traffic it gets. "Opportunity" lives at the intersection of the two — stale **and** still has something to gain.

So the proxy is a composite: a staleness signal (time since the content was last meaningfully updated) combined with a potential signal (current traffic / impressions / ranking position — evidence Google and users still care about the page, or once did).

In [3]:
import pandas as pd

DATA_PATH = "content_refresh_anonymized.csv"  # adjust if running outside work/notebooks/
df = pd.read_csv(DATA_PATH)
print(df.shape)
print(list(df.columns))

# I don't have the exact column names memorized -- data-dictionary.md is the source of truth.
# This just does a keyword scan so I can see candidate columns for each half of the proxy
# instead of guessing names.
staleness_keywords = ["day", "date", "update", "stale", "age", "published", "modified"]
potential_keywords = ["traffic", "click", "impression", "position", "rank", "session", "visit"]

staleness_candidates = [c for c in df.columns if any(k in c.lower() for k in staleness_keywords)]
potential_candidates = [c for c in df.columns if any(k in c.lower() for k in potential_keywords)]

print("\nCandidate staleness columns:", staleness_candidates)
print("Candidate traffic/potential columns:", potential_candidates)

# TODO once I've checked data-dictionary.md: replace these with the real chosen columns,
# scale each 0-1, and combine into one opportunity_score column, e.g.:
# df["opportunity_score"] = 0.5 * staleness_0_1 + 0.5 * potential_0_1


(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Candidate staleness columns: ['pageviews_90d', 'engaged_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'engagement_rate']
Can

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@K.**

There's no ground-truth label for "this page was correctly flagged" sitting in the data, so I can't compute something like accuracy directly. What I can defend is: sort all pages by my opportunity score, take the top K (e.g. top 50), and check what fraction of those a reasonable editor would actually agree are worth refreshing. That fraction is Precision@K.

This is also the same metric the starter pipeline's own baseline-vs-model comparison uses (Precision@50), so it's a defensible, comparable choice — and it directly reflects the real action: an editor isn't going to work through all 30,000 pages, they're going to work down a short priority list, so what matters is whether the *top* of that list is right.

In [4]:
def precision_at_k(scored_df, score_col, label_col, k=50):
    """Fraction of the top-k scored rows that are true positives, by some ground-truth-ish label_col."""
    top_k = scored_df.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

print("METRIC = Precision@50")
print("Once opportunity_score exists, and once I define a face-validity label "
      "(e.g. from the baseline hand-rule in scripts/02_baseline_score.py), "
      "I'll run: precision_at_k(df, 'opportunity_score', 'label_col', k=50)")


METRIC = Precision@50
Once opportunity_score exists, and once I define a face-validity label (e.g. from the baseline hand-rule in scripts/02_baseline_score.py), I'll run: precision_at_k(df, 'opportunity_score', 'label_col', k=50)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page.** Every row in `content_refresh_anonymized.csv` is a single content page on a client's site, described by its staleness and performance signals. My score gets computed per row, i.e. per page.

In [5]:
# df was already loaded in section 2 -- show the unit of analysis directly
print(f"Rows (pages): {len(df)}")
print(f"Columns: {len(df.columns)}")
df.head(10)


Rows (pages): 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule would look like: `if age_in_months > X and traffic > Y: flag it`. That breaks down for two reasons:

1. **Not all pages decay at the same rate.** A fast-moving topic can go stale in a couple of months; an evergreen reference page can stay useful for years. One fixed age threshold either flags evergreen pages that don't need it, or misses fast-decaying pages before it's too late.
2. **It doesn't generalize across very different page types.** A single threshold tuned for one category of page (say, product pages) won't transfer cleanly to another (say, blog posts or landing pages) — they have different baseline traffic levels, different typical lifespans, and the *relationship* between staleness and traffic potential isn't the same shape across them.

A model can learn how staleness and potential interact differently across page types instead of forcing one hand-picked cutoff onto every page.

In [6]:
# quick, honest check of the claim above: does a single fixed rule actually behave consistently
# across page types, or does it over/under-flag some groups? Adjust column names once confirmed
# against data-dictionary.md.
candidate_group_cols = [c for c in df.columns if "type" in c.lower() or "categor" in c.lower()]
print("Candidate page-type/category columns to check this against:", candidate_group_cols)


Candidate page-type/category columns to check this against: ['content_type']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.